# 🧵 Listas Enlazadas — Positional List ADT · Implementación

## Programación III - Lic. en Sistemas

![daniele-levis-pelusi-liW9Gw8et18-unsplash.jpg](images/daniele-levis-pelusi-liW9Gw8et18-unsplash.jpg)
    
Fuente: [Daniele Levis Pelusi en Unsplash](https://unsplash.com/es/fotos/lemur-gris-cerca-del-arbol-liW9Gw8et18?utm_content=creditCopyText&utm_medium=referral&utm_source=unsplash)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap07/03_LinkedLists_ADT_Positional.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, podrás:

1. Entender el ADT de Lista Posicional: concepto de Position y operaciones.
2. Implementar una Lista Posicional sobre una base de lista doblemente enlazada.

---

## ADT Lista Posicional

* **Abstracción General:** A diferencia de las pilas, colas y deques que solo permiten operaciones en los extremos, la **Lista Posicional** es una abstracción más general que permite al usuario referirse a elementos en cualquier lugar de una secuencia y realizar inserciones y eliminaciones arbitrarias.

* **Problema de índices:** Los índices numéricos no son una buena opción para listas enlazadas debido a la ineficiencia del acceso y porque cambian con inserciones/eliminaciones.

* **Abstracción de Posición:** En lugar de nodos directos, que violarían los principios de encapsulación, se introduce una abstracción de posición independiente para indicar la ubicación de un elemento dentro de una lista.

* **Posición:** Una posición/__position__ `p` no se ve afectada por los cambios que se produzcan en otras partes de una lista; la única forma en que una posición deja de ser válida es si se emite un comando explícito para eliminarla. Un objeto `position` es simple y solo soporta `p.element()`. Se expone un objeto **Position** que representa una ubicación válida en la secuencia. 

### Métodos del ADT Lista Posicional

* **Acceso:** `L.first()`, `L.last()`, `L.before(p)`, `L.after(p)`, `L.is_empty()`, `len(L)`, `iter(L)`. Estos métodos devuelven objetos **position**, no elementos.

* **Actualización:** `L.add_first(e)`, `L.add_last(e)`, `L.add_before(p, e)`, `L.add_after(p, e)`, `L.replace(p, e)`, `L.delete(p)`. Todos los métodos de actualización que toman una posición `p` como parámetro verifican que sea una posición válida para la lista `L`.

### Beneficios
- Operar localmente en O(1) dado un objeto `Position`.
- Encapsula detalles del nodo; mantiene invariantes de la estructura.

---

## Implementación de Lista Posicional

La implementación anterior muestra el patrón estándar: construir `PositionalList` heredando de una base doblemente enlazada con centinelas y envolviendo nodos con un objeto `Position`. Esta arquitectura separa claramente la "ubicación lógica" (Position) del "detalle físico" (nodos).

Buenas prácticas:
- Validar posiciones para evitar aliasing o acceso a nodos de otra lista.
- Invalidar nodos eliminados (next = None) para que no puedan reusarse.
- Exponer solo la API de posiciones (no nodos internos).

---

In [ ]:
# Implementación de PositionalList sobre una lista doblemente enlazada con centinelas
class DoublyLinkedBase:
    class _Node:
        __slots__ = ('element', 'prev', 'next')
        def __init__(self, element=None, prev=None, next=None):
            self.element = element; self.prev = prev; self.next = next
    def __init__(self):
        self._header = self._Node(); self._trailer = self._Node()
        self._header.next = self._trailer; self._trailer.prev = self._header
        self._size = 0
    def __len__(self): return self._size
    def is_empty(self): return self._size == 0
    def _insert_between(self, e, predecessor, successor):
        node = self._Node(e, predecessor, successor)
        predecessor.next = node; successor.prev = node
        self._size += 1
        return node
    def _delete_node(self, node):
        p, s = node.prev, node.next
        p.next = s; s.prev = p
        self._size -= 1
        e = node.element
        node.prev = node.next = node.element = None
        return e

class PositionalList(DoublyLinkedBase):
    class Position:
        __slots__ = ('_container','_node')
        def __init__(self, container, node):
            self._container = container 
            self._node = node
        def element(self):
            return self._node.element
        def __eq__(self, other):
            return type(other) is type(self) and other._node is self._node

    # utilidades internas
    def _validate(self, p):
        if not isinstance(p, self.Position):
            raise TypeError('p must be proper Position type')
        if p._container is not self:
            raise ValueError('p does not belong to this container')
        if p._node.next is None:
            raise ValueError('p is no longer valid')
        return p._node
    
    def _make_position(self, node):
        if node is self._header or node is self._trailer:
            return None
        return self.Position(self, node)

    # Accesores de posiciones
    def first(self): return self._make_position(self._header.next)
    def last(self): return self._make_position(self._trailer.prev)
    def before(self, p):
        node = self._validate(p)
        return self._make_position(node.prev)
    def after(self, p):
        node = self._validate(p)
        return self._make_position(node.next)

    # Mutadores basados en posiciones
    def _insert_between_pos(self, e, predecessor, successor):
        node = self._insert_between(e, predecessor, successor)
        return self._make_position(node)
    
    def add_first(self, e): return self._insert_between_pos(e, self._header, self._header.next)
    def add_last(self, e): return self._insert_between_pos(e, self._trailer.prev, self._trailer)
    def add_before(self, p, e):
        node = self._validate(p)
        return self._insert_between_pos(e, node.prev, node)
    def add_after(self, p, e):
        node = self._validate(p)
        return self._insert_between_pos(e, node, node.next)
    def delete(self, p):
        node = self._validate(p)
        return self._delete_node(node)
    def replace(self, p, e):
        node = self._validate(p)
        old = node.element; node.element = e; return old

# Demo rápida
plist = PositionalList()
p1 = plist.add_first(10)
p2 = plist.add_after(p1, 20)
print('first=', plist.first().element(), 'last=', plist.last().element())
plist.replace(p2, 25)
print('after first=', plist.after(p1).element())

### Complejidad
- Dado un Position válido: add_before/after, add_first/last, delete, replace → O(1).
- Recorridos secuenciales por posiciones → O(n).

---

## 🔢 Sorting a Positional List

Ordenar una `PositionalList` puede hacerse con algoritmos in-place que usan movimientos locales (recolocación por posiciones). Un enfoque instructivo es la inserción ordenada (insertion sort) que recorre la lista y reubica cada elemento en su posición correcta a la izquierda.

Idea del algoritmo (Insertion Sort sobre `PositionalList`):
1. 🧭 Mantener una frontera entre la sublista ordenada (izquierda) y la por ordenar (derecha).
2. 🎯 Tomar la primera posición de la parte desordenada (pivot) y desplazarla hacia la izquierda hasta encontrar su lugar.
3. 🔁 Repetir hasta agotar la parte desordenada.

⏱️ Complejidad: O(n^2) en el peor caso; movimientos locales O(1) (add_before/delete).

<div align="center">
<!-- mini-diagrama de la frontera de insertion sort -->
<svg width="620" height="80" xmlns="http://www.w3.org/2000/svg">
  <rect x="10" y="25" width="260" height="30" rx="6" fill="#e6f7ff" stroke="#69c0ff"/>
  <text x="140" y="45" text-anchor="middle" font-family="monospace" font-size="12">Ordenado</text>
  <rect x="280" y="25" width="260" height="30" rx="6" fill="#fff1f0" stroke="#ff7875"/>
  <text x="410" y="45" text-anchor="middle" font-family="monospace" font-size="12">Por ordenar</text>
  <circle cx="280" cy="40" r="6" fill="#ff4d4f"/>
  <text x="305" y="20" font-family="monospace" font-size="12">pivot</text>
</svg>
</div>

---

In [ ]:
def insertion_sort_positional(plist: PositionalList):
    if plist.is_empty():
        return
    marker = plist.first()  # fin de la zona ordenada
    while marker != plist.last():
        pivot = plist.after(marker)         # primer elemento de la zona por ordenar
        value = pivot.element()
        if value >= marker.element():       # ya está en orden, avanzar frontera
            marker = pivot
        else:
            # buscar sitio a la izquierda
            walk = marker
            while walk != plist.first() and plist.before(walk).element() > value:
                walk = plist.before(walk)
            # quitar pivot y reinsertar antes de walk o al inicio
            plist.delete(pivot)
            if walk == plist.first() and plist.first().element() > value:
                plist.add_first(value)
            else:
                plist.add_after(walk, value)

# Demo de ordenación
data = [7, 3, 5, 1, 9, 2]
pl = PositionalList()
last = None
for x in data:
    last = pl.add_last(x)
print('antes:', [pl.first().element(), pl.last().element()])
insertion_sort_positional(pl)
# recorrido para mostrar resultado
out = []
p = pl.first()
while p is not None:
    out.append(p.element())
    p = pl.after(p)
print('ordenado:', out)

## 📊 Case Study: Maintaining Access Frequencies

Problema: mantener una colección de elementos ordenada por la frecuencia de acceso (más accedidos primero).

Operaciones:
- ➕ `access(e)`: incrementa la frecuencia de `e` y reubica su posición.
- 🔝 `top(k)`: devuelve los `k` elementos más accedidos.

Estrategia:
- 📚 Usar una `PositionalList` de registros `(elemento, frecuencia)`.
- 🧲 En `access(e)`, si `e` no está, insertar con `freq=1`; si está, incrementar y moverlo hacia su lugar correcto (insertion sort local, O(r) con r = distancia movida).

⏱️ Complejidad:
- Cada acceso cuesta O(d) donde d es la distancia de desplazamiento (mejor que reordenar completo cada vez).

<div align="center">
<!-- diagrama de ranking -->
<svg width="560" height="96" xmlns="http://www.w3.org/2000/svg">
  <rect x="20" y="20" width="520" height="20" rx="4" fill="#fafafa" stroke="#d9d9d9"/>
  <text x="280" y="35" text-anchor="middle" font-family="monospace" font-size="12">(A,5)  →  (B,4)  →  (C,3)  →  (D,1)</text>
  <text x="280" y="70" text-anchor="middle" font-family="monospace" font-size="12">access(C) ⇒ (C,4) sube por delante de (B,4) según criterio</text>
</svg>
</div>

---

In [ ]:
class FavoritesList:
    """Mantiene elementos con sus frecuencias, ordenados por acceso."""
    class _Item:
        __slots__ = ('value', 'count')
        def __init__(self, value):
            self.value = value; self.count = 0
        def __repr__(self): return f"Item({self.value!r},{self.count})"

    def __init__(self):
        self._data = PositionalList()

    def __len__(self): return len(self._data)

    def is_empty(self): return self._data.is_empty()

    def _find_position(self, e):
        p = self._data.first()
        while p is not None:
            if p.element().value == e:
                return p
            p = self._data.after(p)
        return None

    def access(self, e):
        p = self._find_position(e)
        if p is None:
            p = self._data.add_last(self._Item(e))
        item = p.element(); item.count += 1
        # subir mientras el anterior tenga menor frecuencia
        walk = self._data.before(p)
        while walk is not None and walk.element().count < item.count:
            p = self._data.before(p)
            walk = self._data.before(p)
        # reubicar si cambió su lugar
        # técnica: eliminar e insertar antes de la primera posición con menor freq
        # (si p no es la posición original)
        q = self._data.before(self._data.after(p))  # posición original previa al bucle
        # Para simplicidad, reconstruimos búsqueda de la posición final:
        self._data.delete(self._data.after(q) if q is not None else self._data.first())
        if walk is None:
            self._data.add_first(item)
        else:
            self._data.add_after(walk, item)

    def top(self, k):
        # devolver valores de los k más frecuentes
        res = []
        p = self._data.first()
        i = 0
        while p is not None and i < k:
            res.append((p.element().value, p.element().count))
            p = self._data.after(p); i += 1
        return res

# Demostración rápida
fav = FavoritesList()
for x in ["a","b","a","c","a","b","d","b","b"]:
    fav.access(x)
print('top2:', fav.top(2))

## 🔗 Link-Based vs. 🧱 Array-Based Sequences

Comparación de representaciones para secuencias (listas enlazadas vs. arreglos dinámicos):

- 🔍 Acceso por índice:
  - 🧱 Arreglos: O(1) directo.
  - 🔗 Enlazadas: O(n) (recorrido).
- ✍️ Inserciones/Eliminaciones locales (con posición conocida):
  - 🧱 Arreglos: O(n) por desplazamientos; amortización por redimensionamiento.
  - 🔗 Enlazadas: O(1) (ajuste de punteros) si se tiene la posición/nodo.
- 🧠 Localidad de memoria y caché:
  - 🧱 Arreglos: excelente (contiguidad), mejor rendimiento práctico.
  - 🔗 Enlazadas: dispersión, peor localidad, más saltos de puntero.
- ⚖️ Redimensionamiento:
  - 🧱 Arreglos: ocasional O(n), amortizado O(1) por push/pop al final.
  - 🔗 Enlazadas: no requieren redimensionamiento global.
- 🧰 Soporte para ADTs avanzados:
  - 🔗 Positional List se ajusta naturalmente a listas enlazadas.

📌 Conclusión: elegir según patrón de acceso/actualización. Si prima acceso aleatorio por índice → arreglos. Si predominan inserciones/eliminaciones locales y recorridos secuenciales → enlazadas.

---

## 📌 Resumen 

- Listas Circulares: `tail.next → head`, `rotate()` O(1); modelos cíclicos (round-robin, buffers).
- Doble Enlace + Centinelas: inserciones/eliminaciones uniformes O(1) y código simétrico.
- Positional List ADT: `Position` como abstracción de ubicación; validación e iteración seguras.
- Sorting en PositionalList: insertion sort in-place con movimientos locales O(1); O(n^2) en el peor caso.
- Case Study (frecuencias de acceso): mantener lista ordenada por uso con actualizaciones locales eficientes.
- Link vs Array: índices O(1) y buena localidad en arreglos; actualizaciones locales O(1) y ADT posicional natural en enlazadas.

## ✅ Auto-chequeo
- [ ] ¿Puedo describir la API de `PositionalList` y su complejidad?
- [ ] ¿Sé implementar insertion sort sobre una `PositionalList`?
- [ ] ¿Entiendo cómo mantener una lista ordenada por frecuencia de acceso?
- [ ] ¿Puedo justificar cuándo elegir listas enlazadas vs. arreglos?




---

## 📖 Referencias Bibliográficas

**[DSAP]** Goodrich, M. T., Tamassia, R., & Goldwasser, M. H. (2013). *Data Structures and Algorithms in Python*. John Wiley & Sons.

- Capítulo 7: Linked Lists
- Sección 7.4: The Positional List ADT
- Sección 7.5: Sorting a Positional List
- Sección 7.7: Link-Based vs. Array-Based Sequences

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2025 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This material is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)